# Cone Detection - YOLOv8 Training
This notebook trains a YOLOv8n model to detect and classify cones (blue, orange, large orange, yellow) using a GPU on Google Colab.

## 1. Install Dependencies & Clone Repo

In [ ]:
!pip install ultralytics -q

In [ ]:
!git clone https://github.com/lior4715/Cone-Detection.git
%cd Cone-Detection

## 2. Verify GPU is Available

In [ ]:
import torch
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 3. Fix data.yaml Paths for Colab

In [ ]:
import os
import yaml

data_yaml_path = "datasets/data.yaml"
datasets_dir = os.path.abspath("datasets")

with open(data_yaml_path, "r") as f:
    data = yaml.safe_load(f)

data["path"] = datasets_dir
data["train"] = "train/images"
data["val"] = "valid/images"

with open(data_yaml_path, "w") as f:
    yaml.dump(data, f, default_flow_style=False)

print(f"Updated data.yaml path to: {datasets_dir}")
print(f"Train images: {len(os.listdir('datasets/train/images'))}")
print(f"Valid images: {len(os.listdir('datasets/valid/images'))}")

## 4. Train the Model

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

model.train(
    data="datasets/data.yaml",
    epochs=50,
    imgsz=640,
    project="runs/detect",
    name="new_training",
    exist_ok=True,
)

## 5. View Training Results

In [ ]:
from IPython.display import Image, display

results_dir = "runs/detect/new_training"

for img_name in ["results.png", "confusion_matrix.png", "val_batch0_pred.jpg"]:
    img_path = os.path.join(results_dir, img_name)
    if os.path.exists(img_path):
        print(f"\n{img_name}:")
        display(Image(filename=img_path, width=800))

## 6. Run Inference on Video

In [ ]:
weights_path = "runs/detect/new_training/weights/best.pt"

model = YOLO(weights_path)

results = model.predict(
    source="videos/fsd1.mp4",
    save=True,
    conf=0.25,
)

print("Inference complete! Check runs/detect/predict/ for the output video.")

## 7. Download Trained Weights
Download `best.pt` so you can use it locally for inference.

In [ ]:
from google.colab import files

files.download("runs/detect/new_training/weights/best.pt")